# CHIRPS x OSM

- replacing the placeholder `HIGHWAY_SURFACE_RISK` table used in `graph-representation/03_pulling_real_graph.ipynb`
with a rainfall-aware version.

Recap of the decisions this notebook implements:
- **Accumulation window:** rolling 3-day sum of daily CHIRPS precipitation (mm), per pixel. (To Review - we may need to consider that is accumulation in terms of draining water in different soils)
- **Rain risk thresholds:** `<5mm -> 0.0`, `5-15mm -> 0.3`, `15-30mm -> 0.6`, `>=30mm -> 1.0`.
- **Road rain vulnerability:** how much a given `road_type` is affected by rain
  (paved roads barely, tracks a lot).
- **Final formula:** `surface_risk = min(1.0, base_road_risk + vulnerability[road_type] * rain_risk)`.

In [1]:
%pip install osmnx networkx geopandas shapely folium earthengine-api pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ee

In [3]:
ee.Authenticate() # TODO: we need to obtain credentials with a KV


Successfully saved authorization token.


In [4]:
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5') # TODO: we need to obtain credentials with a KV

In [5]:
import folium

### Core implementation

In [6]:
from park_road_network import ParkRoadNetwork

PLACE_NAME = "Tarangire National Park, Tanzania"
RAIN_WINDOW_START_DATE = "2024-12-01"
RAIN_WINDOW_END_DATE = "2024-12-30"

nodes, edges = ParkRoadNetwork(PLACE_NAME).load() #TODO: Check the conditions for this. and also better attributes
print(f"Vehicle-usable edges: {len(edges)}")

Vehicle-usable edges: 2207


In [7]:
print(type(nodes), type(edges))

<class 'geopandas.geodataframe.GeoDataFrame'> <class 'geopandas.geodataframe.GeoDataFrame'>


In [8]:
print("-----------Nodes-----------")
print(nodes.head())

print("-----------Edges-----------")
print(edges.head())

-----------Nodes-----------
                  y          x  street_count highway  \
osmid                                                  
460425854 -4.085310  36.116738             3     NaN   
460425855 -4.085099  36.116907             3     NaN   
460425857 -4.082869  36.118470             3     NaN   
618532432 -4.150237  36.116413             3     NaN   
618532444 -4.097150  36.119965             3     NaN   

                            geometry  
osmid                                 
460425854  POINT (36.11674 -4.08531)  
460425855   POINT (36.11691 -4.0851)  
460425857  POINT (36.11847 -4.08287)  
618532432  POINT (36.11641 -4.15024)  
618532444  POINT (36.11997 -4.09715)  
-----------Edges-----------
                              osmid      access     road_type maxspeed  \
u         v          key                                                 
460425854 4681959723 0    474196213  permissive  unclassified       50   
          460425855  0    474196213  permissive  unclass

In [9]:
from chirps_provider import CHIRPSProvider

rain_provider = CHIRPSProvider()
pixel_grid = rain_provider.frame_grid(edges.total_bounds) #Frame CRIPS grid with the coordinates from OSMnx

print(pixel_grid.head())

rain_pixels, image_collection = rain_provider.get_accumulated_precipitation(
    pixel_grid,
    start_date=RAIN_WINDOW_START_DATE, #TODO: dynamic selection and avoid str as input
    end_date=RAIN_WINDOW_END_DATE
)

print(f"Pixels: {len(rain_pixels)}")
print(rain_pixels["rain_mm"].describe())

   pixel_id                                           geometry
0         0  POLYGON ((35.9 -4.55, 35.9 -4.5, 35.85 -4.5, 3...
1         1  POLYGON ((35.9 -4.5, 35.9 -4.45, 35.85 -4.45, ...
2         2  POLYGON ((35.9 -4.45, 35.9 -4.4, 35.85 -4.4, 3...
3         3  POLYGON ((35.9 -4.4, 35.9 -4.35, 35.85 -4.35, ...
4         4  POLYGON ((35.9 -4.35, 35.9 -4.3, 35.85 -4.3, 3...
Pixels: 180
count    180.000000
mean       6.808518
std        0.787006
min        5.204319
25%        6.236650
50%        6.673927
75%        7.309522
max        8.902940
Name: rain_mm, dtype: float64


In [10]:
from IPython.display import Image, display

serengeti = ee.Geometry.Rectangle([
    33.8, -3.5,
    35.3, -1.2,
])

url = image_collection.getVideoThumbURL({
    "region": serengeti,
    "dimensions": 800,
    "framesPerSecond": 3,
    "min": 0,
    "max": 50,
    "palette": ["white", "blue", "purple"],
})

display(Image(url=url))

In [13]:
from edge_risk_enricher import EdgeRiskEnricher

enricher = EdgeRiskEnricher()
edges_enriched = enricher.enrich(edges, rain_pixels) #Enrich with risk_* attributes in all edges.

print(edges_enriched[[
    "road_type",
    "rain_mm",
    "surface_risk"
]].head(10))

print("\nsurface_risk distribution:")
print(edges_enriched["surface_risk"].describe())

print("\nrain_mm distribution:")
print(edges_enriched["rain_mm"].describe())

                             road_type   rain_mm  surface_risk
u         v          key                                      
460425854 4681959723 0    unclassified  6.758968          0.31
          460425855  0    unclassified  6.758968          0.31
          4681960245 0           track  6.758968          0.45
460425855 460425854  0    unclassified  6.758968          0.31
          4681960245 0           track  6.758968          0.45
          460425857  0    unclassified  6.758968          0.31
460425857 4681960250 0           track  6.758968          0.45
          460425855  0    unclassified  6.758968          0.31
          2196664391 0    unclassified  7.264799          0.31
618532432 4660249470 0           track  6.867643          0.45

surface_risk distribution:
count    2207.000000
mean        0.390376
std         0.087115
min         0.170000
25%         0.310000
50%         0.450000
75%         0.450000
max         0.450000
Name: surface_risk, dtype: float64

rain_mm dist

### Visualizing edge cost

In [15]:
center_lat, center_lon = nodes["y"].mean(), nodes["x"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

SURFACE_RISK_COLORS = {
    (0.0, 0.25): "#2ecc71",
    (0.25, 0.5): "#f39c12",
    (0.5, 0.75): "#e74c3c",
    (0.75, 1.01): "#7b241c",
}

def get_risk_color(surface_risk: float) -> str:
    for (low, high), color in SURFACE_RISK_COLORS.items():
        if low <= surface_risk < high:
            return color
    return "#7b241c"

for _, row in edges_enriched.iterrows():
    coords = [(lat, lon) for lon, lat in row["geometry"].coords]
    road_type = row["road_type"]
    rain_mm = row["rain_mm"]
    travel_time = row["travel_time_m"]

    print(f"Road type: {road_type}; rain mm: {rain_mm}; surface risk: {row["surface_risk"]}; travel time: {travel_time}")

    surface_risk = row["surface_risk"]

    folium.PolyLine(
        locations=coords,
        color=get_risk_color(surface_risk),
        weight=3,
        opacity=0.8,
        popup=(
            f"highway: {road_type}<br>"
            f"travel_time: {travel_time}<br>"
            f"rain_mm(accumualte over x (start - end) days): {rain_mm:.1f}<br>"
            f"surface_risk: {surface_risk:.2f}<br>"
            f"surface_risk: {surface_risk:.2f}"
        ),
    ).add_to(m)

m


Road type: unclassified; rain mm: 6.758968353271484; surface risk: 0.31; travel time: 0.022192262387747266
Road type: unclassified; rain mm: 6.758968353271484; surface risk: 0.31; travel time: 0.036042562668318075
Road type: track; rain mm: 6.758968353271484; surface risk: 0.44999999999999996; travel time: 0.030258153192196807
Road type: unclassified; rain mm: 6.758968353271484; surface risk: 0.31; travel time: 0.036042562668318075
Road type: track; rain mm: 6.758968353271484; surface risk: 0.44999999999999996; travel time: 0.017009389461004824
Road type: unclassified; rain mm: 6.758968353271484; surface risk: 0.31; travel time: 0.37398566556556684
Road type: track; rain mm: 6.758968353271484; surface risk: 0.44999999999999996; travel time: 0.34781434269989947
Road type: unclassified; rain mm: 6.758968353271484; surface risk: 0.31; travel time: 0.373985665565567
Road type: unclassified; rain mm: 7.264798641204834; surface risk: 0.31; travel time: 8.250063151227131
Road type: track; rai

### Next steps

- Validate the rain-risk thresholds and road-vulnerability multipliers against real
  road-condition reports, once available — they're reasoned estimates, not fitted.
- `RAIN_WINDOW_END_DATE` is hardcoded to a date from the eda exploration; wire it to
  "most recent date CHIRPS has published" for anything beyond notebook experimentation.
- Feed `edges_enriched` (with its `cost` column) into a shortest-path routine
  (e.g. `networkx.shortest_path` with `weight="cost"`) to get an actual risk-aware route.